# NB30: USGS λ-vs-β Control Sweep

Tests whether a metal KO's unconditional Pagel's λ predicts its PGLS association
strength (|β|) with USGS-measured soil metal concentrations, across a grid of
environmental control combinations.

**Control combinations tested (12):**
1. None (baseline)
2. pH
3. SOM
4. Temperature
5. pH + SOM
6. pH + Temperature
7. SOM + Temperature
8. pH + SOM + Temperature
9. |Latitude| (geographic)
10. pH + |Latitude|
11. pH + SOM + |Latitude|
12. pH + SOM + Temperature + |Latitude| (full panel)

**Output:**
- `nb30_usgs_control_sweep.parquet` — 35 KOs × 7 metals × 12 control combos
- `nb30_lambda_beta_summary.csv` — Spearman ρ(λ, |β|) per metal × control combo
- Figures: heatmap of ρ values; scatter plots for interesting combos


In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

ROOT  = Path('/home/hmacgregor/BERIL-research-observatory')
DATA  = ROOT / 'projects/comprehensive_metal_ecology/data'
FIGS  = ROOT / 'projects/comprehensive_metal_ecology/figures'
TREE_PATH = DATA / 'gtdb_bac_genus_pruned.tree'

sys.path.insert(0, str(ROOT / 'tools'))
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H, grid_h
apply_style()

sys.path.insert(0, str(ROOT / 'projects/comprehensive_metal_ecology/scripts'))
from pgls_utils import load_tree, build_vcv, _optimise_lambda, _gls_fit

print('Setup done.')


Setup done.


In [2]:
# ── Load static datasets (same as NB27) ───────────────────────────────────────
print('Loading data...')

phylo_lam  = pd.read_csv(DATA / 'phylo_d_all_ko.csv')              # 276 metal KOs, unconditional λ
curated    = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')       # KO metadata
genus_env  = pd.read_csv(DATA / 'genus_lat_env_covariates.csv')    # env covariates
spark      = pd.read_csv(DATA / '01_genus_ko_density_spark.csv')   # genome stats
genus_usgs = pd.read_csv(DATA / 'nb27_genus_usgs_means.csv')       # USGS genus means
csu_beta   = pd.read_csv(DATA / 'per_ko_lambda_csu_mobility.csv')  # 35 fitted KOs

nb25 = pd.read_parquet(DATA / 'nb25_ko_presence_matrix.parquet')  # metal KO presence
nb25['genus_lower'] = nb25['genus_lower'].str.replace('g__', '', regex=False)

tier12     = curated[curated['evidence_tier'].isin(['Tier 1', 'Tier 2'])]
FITTED_KOS = sorted(csu_beta['ko_id'].unique())   # 35 KOs

# Build KO density table
density_base = (
    nb25[nb25['ko'].isin(FITTED_KOS)]
    .merge(spark[['genus_lower', 'n_genomes', 'mean_genome_mb']], on='genus_lower', how='inner')
)
density_base['density'] = (
    density_base['n_genomes_with_ko'] /
    (density_base['n_genomes'] * density_base['mean_genome_mb'])
)

# Load phylogenetic tree once (DendroPy API)
tree = load_tree(str(TREE_PATH))
tree_labels = {t.label.replace(' ', '_').lower() for t in tree.taxon_namespace}
print(f'Tree loaded: {len(tree_labels):,} taxa')

USGS_RAW_COLS  = [c for c in genus_usgs.columns if c.startswith('mean_usgs_raw_')]
USGS_METALS    = [c.replace('mean_usgs_raw_', '') for c in USGS_RAW_COLS]
print(f'USGS metals: {USGS_METALS}')
print(f'Fitted KOs:  {len(FITTED_KOS)}')
print(f'Env covariate columns: {list(genus_env.columns)}')


Loading data...
Tree loaded: 2,283 taxa
USGS metals: ['as', 'cd', 'cr', 'cu', 'ni', 'pb', 'zn']
Fitted KOs:  35
Env covariate columns: ['genus_lower', 'n_samples', 'median_lat', 'median_lon', 'lat_abs', 'median_era5_temp_C', 'median_temp_range_C', 'median_soil_ph', 'median_georoc_metal_index', 'median_cmmi_nearest_km', 'georoc_Cu_log', 'georoc_Ni_log', 'georoc_Zn_log', 'georoc_Co_log', 'georoc_Pb_log', 'georoc_Cr_log', 'median_soil_moisture', 'median_soil_som']


In [3]:
# ── PGLS helper (canonical NB27 version) ──────────────────────────────────────
def run_pgls_fast(df, response_col, predictor_cols, taxon_col='genus_lower', min_n=30):
    keep = df[taxon_col].str.replace(' ', '_').str.lower().isin(tree_labels) & df[response_col].notna()
    for pc in predictor_cols:
        keep = keep & df[pc].notna()
    sub = df[keep].copy()
    sub['_taxon'] = sub[taxon_col].str.replace(' ', '_').str.lower()
    sub = sub.drop_duplicates('_taxon')
    n = len(sub)
    if n < min_n:
        return None
    taxa = sub['_taxon'].tolist()
    y    = sub[response_col].values.astype(float)
    X    = np.column_stack([np.ones(n)] + [sub[pc].values.astype(float) for pc in predictor_cols])
    V    = build_vcv(tree, taxa)
    lam, _ = _optimise_lambda(y, X, V)
    ll, sigma2, betas, betas_se, _, _ = _gls_fit(y, X, V, lam)
    t_stats = betas / np.where(betas_se > 0, betas_se, np.nan)
    df_resid = n - len(betas)
    p_values = 2 * stats.t.sf(np.abs(t_stats), df=df_resid)
    return dict(n=n, lambda_est=lam, betas=betas, SEs=betas_se,
                t_stats=t_stats, p_values=p_values,
                beta=betas[1] if len(betas) > 1 else betas[0],
                SE=betas_se[1] if len(betas_se) > 1 else betas_se[0],
                p_value=p_values[1] if len(p_values) > 1 else p_values[0])


def zscore(x):
    mu, sd = np.nanmean(x), np.nanstd(x, ddof=1)
    return (x - mu) / sd if sd > 0 else x - mu


# ── Control combinations ───────────────────────────────────────────────────────
CONTROL_COMBOS = [
    ('None',                       []),
    ('pH',                         ['ph_z']),
    ('SOM',                        ['som_z']),
    ('Temp',                       ['temp_z']),
    ('pH + SOM',                   ['ph_z', 'som_z']),
    ('pH + Temp',                  ['ph_z', 'temp_z']),
    ('SOM + Temp',                 ['som_z', 'temp_z']),
    ('pH + SOM + Temp',            ['ph_z', 'som_z', 'temp_z']),
    ('|Lat|',                      ['lat_z']),
    ('pH + |Lat|',                 ['ph_z', 'lat_z']),
    ('pH + SOM + |Lat|',           ['ph_z', 'som_z', 'lat_z']),
    ('pH + SOM + Temp + |Lat|',    ['ph_z', 'som_z', 'temp_z', 'lat_z']),
]

COMBO_LABELS = [c[0] for c in CONTROL_COMBOS]
print(f'{len(CONTROL_COMBOS)} control combinations defined.')


12 control combinations defined.


In [4]:
# ── PGLS sweep: 35 KOs × 7 metals × 12 control combos ────────────────────────
SWEEP_PATH = DATA / 'nb30_usgs_control_sweep.parquet'

if SWEEP_PATH.exists():
    print('Loading cached sweep results...')
    sweep = pd.read_parquet(SWEEP_PATH)
else:
    print(f'Running PGLS sweep: {len(FITTED_KOS)} KOs × {len(USGS_METALS)} metals × {len(CONTROL_COMBOS)} combos...')
    print(f'  = {len(FITTED_KOS) * len(USGS_METALS) * len(CONTROL_COMBOS)} total fits')

    # Env covariate lookup table
    env_lookup = genus_env[['genus_lower', 'median_soil_ph', 'median_soil_som',
                             'median_era5_temp_C', 'lat_abs']].copy()
    env_lookup = env_lookup.dropna()

    rows = []
    n_done = 0
    n_total = len(FITTED_KOS) * len(USGS_METALS)

    for ko_id in FITTED_KOS:
        density_ko = density_base[density_base['ko'] == ko_id][['genus_lower', 'density']].copy()
        if len(density_ko) == 0:
            continue

        # One merge per KO: density × USGS means × all env covariates
        merged = (
            density_ko
            .merge(genus_usgs, on='genus_lower', how='inner')
            .merge(env_lookup, on='genus_lower', how='inner')
        )

        # Pre-standardize env covariates (done once per KO merge)
        if len(merged) > 5:
            merged['ph_z']      = zscore(merged['median_soil_ph'])
            merged['som_z']     = zscore(merged['median_soil_som'])
            merged['temp_z']    = zscore(merged['median_era5_temp_C'])
            merged['lat_z']     = zscore(merged['lat_abs'])
            merged['density_z'] = zscore(merged['density'])

        meta_rows = tier12[tier12['KO'] == ko_id]
        gene_name   = meta_rows['gene_name'].values[0]      if len(meta_rows) > 0 else ''
        subcategory = meta_rows['primary_category'].values[0] if len(meta_rows) > 0 else ''

        for metal in USGS_METALS:
            metal_col = f'mean_usgs_raw_{metal}'
            all_needed = [metal_col, 'density', 'median_soil_ph', 'median_soil_som',
                          'median_era5_temp_C', 'lat_abs']
            sub = merged.dropna(subset=all_needed).copy()
            if len(sub) < 20:
                n_done += 1
                continue

            sub['metal_log'] = np.log1p(sub[metal_col])
            sub['metal_z']   = zscore(sub['metal_log'])

            for combo_label, extra_preds in CONTROL_COMBOS:
                predictors = ['density_z'] + extra_preds
                result = run_pgls_fast(sub, 'metal_z', predictors)
                if result:
                    rows.append({
                        'ko_id':       ko_id,
                        'metal':       metal,
                        'gene_name':   gene_name,
                        'subcategory': subcategory,
                        'controls':    combo_label,
                        'lambda_est':  result['lambda_est'],
                        'beta':        result['beta'],
                        'SE':          result['SE'],
                        'p_value':     result['p_value'],
                        'n_genera':    result['n'],
                    })

            n_done += 1
            if n_done % 50 == 0:
                print(f'  {n_done}/{n_total} KO-metal pairs done...')

    sweep = pd.DataFrame(rows)
    sweep.attrs = {}
    sweep.to_parquet(SWEEP_PATH, index=False)
    print(f'Sweep complete: {len(sweep):,} rows saved.')

print(f'Sweep shape: {sweep.shape}')
print(f'Controls: {sorted(sweep["controls"].unique())}')
print(f'Metals:   {sorted(sweep["metal"].unique())}')


Running PGLS sweep: 35 KOs × 7 metals × 12 combos...
  = 2940 total fits


  50/245 KO-metal pairs done...


  100/245 KO-metal pairs done...


  150/245 KO-metal pairs done...


  200/245 KO-metal pairs done...


Sweep complete: 2,256 rows saved.
Sweep shape: (2256, 10)
Controls: ['None', 'SOM', 'SOM + Temp', 'Temp', 'pH', 'pH + SOM', 'pH + SOM + Temp', 'pH + SOM + Temp + |Lat|', 'pH + SOM + |Lat|', 'pH + Temp', 'pH + |Lat|', '|Lat|']
Metals:   ['as', 'cd', 'cr', 'cu', 'ni', 'pb', 'zn']


In [5]:
# ── Join unconditional λ with sweep results ────────────────────────────────────
phylo_lam_sub = phylo_lam[phylo_lam['ko_id'].isin(FITTED_KOS)][['ko_id', 'lambda']].rename(
    columns={'lambda': 'lambda_uncond'})

sweep_lam = sweep.merge(phylo_lam_sub, on='ko_id', how='inner')
sweep_lam['abs_beta'] = np.abs(sweep_lam['beta'])
sweep_lam['sig']      = sweep_lam['p_value'] < 0.05

print(f'Joined rows: {len(sweep_lam):,}')
print(f'KOs with λ: {sweep_lam["ko_id"].nunique()}')

Joined rows: 2,256
KOs with λ: 27


In [6]:
# ── Spearman ρ(λ_uncond, |β_USGS|) per metal × control combo ─────────────────
summary_rows = []

print('Spearman ρ(λ_uncond, |β_USGS|) — all metal × control combinations:')
print(f'{"Metal":6s}  {"Controls":<28s}  {"ρ":>7s}  {"p":>10s}  {"n":>4s}')
print('-' * 62)

for (metal, combo_label), grp in sweep_lam.groupby(['metal', 'controls']):
    if len(grp) < 5:
        continue
    rho, p = stats.spearmanr(grp['lambda_uncond'], grp['abs_beta'])
    n_sig = grp['sig'].sum()
    summary_rows.append({
        'metal': metal, 'controls': combo_label,
        'rho': rho, 'p_value': p, 'n_kos': len(grp),
        'n_sig_kos': n_sig,
    })
    marker = '*' if p < 0.05 else ' '
    print(f'{metal.upper():6s}  {combo_label:<28s}  {rho:+7.3f}  {p:10.3e}  {len(grp):>4d} {marker}')

summary = pd.DataFrame(summary_rows)
summary.to_csv(DATA / 'nb30_lambda_beta_summary.csv', index=False)
print(f'\nSaved nb30_lambda_beta_summary.csv ({len(summary)} rows)')

n_sig_total = (summary['p_value'] < 0.05).sum()
n_total_tests = len(summary)
print(f'\nSignificant (p<0.05): {n_sig_total}/{n_total_tests} combinations')
print(f'Bonferroni threshold: p < {0.05/n_total_tests:.4f}')
n_bonf = (summary['p_value'] < 0.05/n_total_tests).sum()
print(f'Surviving Bonferroni: {n_bonf}/{n_total_tests}')

Spearman ρ(λ_uncond, |β_USGS|) — all metal × control combinations:
Metal   Controls                            ρ           p     n
--------------------------------------------------------------
AS      None                           +0.092   6.496e-01    27  
AS      SOM                            +0.186   3.524e-01    27  
AS      SOM + Temp                     +0.206   3.018e-01    27  
AS      Temp                           +0.155   4.399e-01    27  
AS      pH                             +0.049   8.088e-01    27  
AS      pH + SOM                       +0.078   7.007e-01    27  
AS      pH + SOM + Temp                +0.142   4.810e-01    27  
AS      pH + SOM + Temp + |Lat|        +0.005   9.807e-01    27  
AS      pH + SOM + |Lat|               +0.216   2.789e-01    27  
AS      pH + Temp                      +0.092   6.474e-01    27  
AS      pH + |Lat|                     +0.223   2.626e-01    27  
AS      |Lat|                          +0.245   2.184e-01    27  
CD      None  

In [7]:
# ── Figure 1: ρ heatmap — metals × control combos ─────────────────────────────
pivot_rho = summary.pivot(index='metal', columns='controls', values='rho')
pivot_p   = summary.pivot(index='metal', columns='controls', values='p_value')

# Preserve CONTROL_COMBOS order
col_order = [c[0] for c in CONTROL_COMBOS if c[0] in pivot_rho.columns]
pivot_rho = pivot_rho[col_order]
pivot_p   = pivot_p[col_order]

metals_order = sorted(pivot_rho.index)
pivot_rho = pivot_rho.loc[metals_order]
pivot_p   = pivot_p.loc[metals_order]

fig, ax = plt.subplots(figsize=(FIGW['full'], ROW_H * 1.2))

vmax = 0.6
im = ax.imshow(pivot_rho.values, aspect='auto', cmap='RdBu_r',
               vmin=-vmax, vmax=vmax)

# Annotate cells with ρ value; bold/star if significant
for i, metal in enumerate(metals_order):
    for j, ctrl in enumerate(col_order):
        rho_val = pivot_rho.loc[metal, ctrl]
        p_val   = pivot_p.loc[metal, ctrl]
        if pd.isna(rho_val):
            continue
        marker = '**' if p_val < 0.01 else ('*' if p_val < 0.05 else '')
        txt = f'{rho_val:+.2f}{marker}'
        color = 'white' if abs(rho_val) > 0.35 else 'black'
        ax.text(j, i, txt, ha='center', va='center',
                fontsize=7, color=color,
                fontweight='bold' if marker else 'normal')

ax.set_xticks(range(len(col_order)))
ax.set_xticklabels(col_order, rotation=40, ha='right', fontsize=7)
ax.set_yticks(range(len(metals_order)))
ax.set_yticklabels([m.upper() for m in metals_order], fontsize=8)
ax.set_xlabel('Control combination', fontsize=9)
ax.set_ylabel('USGS metal', fontsize=9)

plt.colorbar(im, ax=ax, label="Spearman ρ (λ_uncond ~ |β_USGS|)", shrink=0.7)
fig.suptitle('NB30: λ vs |β_USGS| across control combinations\n(*p<0.05, **p<0.01)', y=1.02)
save(fig, FIGS / 'nb30_F1_lambda_beta_heatmap')
print('Saved nb30_F1_lambda_beta_heatmap.pdf')

Saved nb30_F1_lambda_beta_heatmap.pdf


In [8]:
# ── Figure 2: ρ trajectory — how each metal's ρ moves as controls are added ───
# Show each metal as a line across the 12 control combos (ordered by complexity)

METAL_COLORS_NB30 = dict(zip(USGS_METALS, PALETTE[:len(USGS_METALS)]))

fig, ax = plt.subplots(figsize=(FIGW['full'], ROW_H))

x_pos = list(range(len(col_order)))

for metal in metals_order:
    rho_vals = [pivot_rho.loc[metal, ctrl] if ctrl in pivot_rho.columns else np.nan
                for ctrl in col_order]
    p_vals   = [pivot_p.loc[metal, ctrl] if ctrl in pivot_p.columns else 1.0
                for ctrl in col_order]
    col = METAL_COLORS_NB30.get(metal, '#7f7f7f')
    ax.plot(x_pos, rho_vals, '-o', color=col, lw=1.2, ms=5, label=metal.upper())
    # Mark significant points
    for xi, (rho_v, p_v) in enumerate(zip(rho_vals, p_vals)):
        if not np.isnan(rho_v) and p_v < 0.05:
            ax.plot(xi, rho_v, 'o', color=col, ms=9,
                    markeredgecolor='k', markeredgewidth=0.8)

ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xticks(x_pos)
ax.set_xticklabels(col_order, rotation=40, ha='right', fontsize=7)
ax.set_ylabel("Spearman ρ (λ_uncond ~ |β_USGS|)", fontsize=9)
ax.set_xlabel('Control combination', fontsize=9)
ax.legend(fontsize=8, ncol=4, loc='upper right')
ax.set_title('ρ trajectory across control combinations\n(filled circles = p<0.05)', fontsize=10)
grid_h(ax)
fig.suptitle('NB30 Fig 2: λ-β ρ trajectory per metal as controls added', y=1.02)
save(fig, FIGS / 'nb30_F2_rho_trajectory')
print('Saved nb30_F2_rho_trajectory.pdf')

Saved nb30_F2_rho_trajectory.pdf


In [9]:
# ── Figure 3: Scatter plots for the most significant metal × control combos ────
# Identify top-5 combinations by absolute ρ

CAT_ORDER = ['Resistance/Detoxification', 'Transport/Homeostasis',
             'Cofactor Biosynthesis', 'Sensing/Regulation',
             'Metal-dependent Metabolism', 'Unknown']
CAT_COLORS = dict(zip(CAT_ORDER, PALETTE))

top_combos = (
    summary
    .assign(abs_rho=lambda d: d['rho'].abs())
    .sort_values('abs_rho', ascending=False)
    .head(6)
    [['metal', 'controls', 'rho', 'p_value']]
)
print('Top 6 metal × control combos by |ρ|:')
print(top_combos.to_string(index=False))

n_panels = len(top_combos)
n_cols = 3
n_rows = (n_panels + n_cols - 1) // n_cols

fig, axs = plt.subplots(n_rows, n_cols, figsize=(FIGW['full'], ROW_H * n_rows))
axs_flat = axs.flatten() if hasattr(axs, 'flatten') else [axs]

for ax, (_, row) in zip(axs_flat, top_combos.iterrows()):
    metal   = row['metal']
    ctrl    = row['controls']
    sub = sweep_lam[(sweep_lam['metal'] == metal) &
                    (sweep_lam['controls'] == ctrl)].copy()

    for cat in CAT_ORDER:
        ss = sub[sub['subcategory'] == cat]
        if len(ss) == 0:
            continue
        ax.scatter(ss['lambda_uncond'], ss['abs_beta'],
                   color=CAT_COLORS.get(cat, '#7f7f7f'), s=30,
                   alpha=0.8, label=cat.split('/')[0],
                   edgecolors='k', linewidths=0.4)
    # Sig markers
    sig = sub[sub['sig']]
    if len(sig) > 0:
        ax.scatter(sig['lambda_uncond'], sig['abs_beta'],
                   facecolors='none', edgecolors='k', s=80, linewidths=1.2, zorder=5)

    rho, p = row['rho'], row['p_value']
    ax.annotate(f'ρ={rho:+.3f}\np={p:.2e}\nn={len(sub)}',
                xy=(0.05, 0.95), xycoords='axes fraction', va='top', fontsize=8)
    ax.set_xlabel("Unconditional Pagel's λ", fontsize=9)
    ax.set_ylabel('|β_USGS|', fontsize=9)
    ax.set_title(f'{metal.upper()} | {ctrl}', fontsize=9)
    grid_h(ax)

# Legend on first panel
axs_flat[0].legend(fontsize=7, loc='upper right')

for ax in axs_flat[n_panels:]:
    ax.set_visible(False)

fig.suptitle('NB30 Fig 3: Top 6 metal × control combos (λ vs |β_USGS|)', y=1.02)
save(fig, FIGS / 'nb30_F3_top_scatter')
print('Saved nb30_F3_top_scatter.pdf')

Top 6 metal × control combos by |ρ|:
metal                controls      rho  p_value
   zn              SOM + Temp 0.531136 0.004363
   zn pH + SOM + Temp + |Lat| 0.482295 0.010843
   cu                      pH 0.481074 0.011075
   pb                     SOM 0.475580 0.012169
   pb                pH + SOM 0.439560 0.021783
   zn         pH + SOM + Temp 0.435897 0.023036


Saved nb30_F3_top_scatter.pdf


In [10]:
# ── Figure 4: β attenuation — how much do controls shrink individual KO β? ────
# For each metal: compare β(None) vs β(pH+SOM+Temp+|Lat|) across KOs
# Shows which KOs are robust vs confound-driven

base_label = 'None'
full_label = 'pH + SOM + Temp + |Lat|'

base_df = sweep_lam[sweep_lam['controls'] == base_label][['ko_id', 'metal', 'beta', 'lambda_uncond', 'subcategory']].rename(
    columns={'beta': 'beta_base'})
full_df = sweep_lam[sweep_lam['controls'] == full_label][['ko_id', 'metal', 'beta']].rename(
    columns={'beta': 'beta_full'})

cmp = base_df.merge(full_df, on=['ko_id', 'metal'], how='inner')
cmp['abs_base'] = cmp['beta_base'].abs()
cmp['abs_full'] = cmp['beta_full'].abs()

rho_cmp, p_cmp = stats.spearmanr(cmp['abs_base'], cmp['abs_full'])
print(f'β(None) vs β(full controls) across all KO-metal pairs:')
print(f'  Spearman ρ = {rho_cmp:+.3f}  p = {p_cmp:.3e}  n = {len(cmp)}')

metals_sorted = sorted(cmp['metal'].unique())
n_cols = 4
n_rows = (len(metals_sorted) + n_cols - 1) // n_cols

fig, axs = plt.subplots(n_rows, n_cols, figsize=(FIGW['full'], ROW_H * n_rows))
axs_flat = axs.flatten() if hasattr(axs, 'flatten') else [axs]

print('\nβ stability (None vs full controls) per metal:')
for ax, metal in zip(axs_flat, metals_sorted):
    sub = cmp[cmp['metal'] == metal]
    if len(sub) < 3:
        ax.set_visible(False)
        continue
    r, p = stats.spearmanr(sub['abs_base'], sub['abs_full'])
    print(f'  {metal.upper():3s}: ρ={r:+.3f}  p={p:.3e}  n={len(sub)}')

    for cat in CAT_ORDER:
        ss = sub[sub['subcategory'] == cat]
        if len(ss) == 0:
            continue
        ax.scatter(ss['abs_base'], ss['abs_full'],
                   color=CAT_COLORS.get(cat, '#7f7f7f'), s=25,
                   alpha=0.8, edgecolors='k', linewidths=0.4)

    lim = max(sub['abs_base'].max(), sub['abs_full'].max()) * 1.05
    ax.plot([0, lim], [0, lim], color='gray', lw=0.8, ls='--')
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.annotate(f'ρ={r:+.2f}\np={p:.2e}', xy=(0.05, 0.95),
                xycoords='axes fraction', va='top', fontsize=8)
    ax.set_xlabel('|β| no controls', fontsize=9)
    ax.set_ylabel('|β| full controls', fontsize=9)
    ax.set_title(metal.upper(), fontsize=10)
    grid_h(ax)

for ax in axs_flat[len(metals_sorted):]:
    ax.set_visible(False)

fig.suptitle('NB30 Fig 4: β stability — no controls vs full control panel', y=1.02)
save(fig, FIGS / 'nb30_F4_beta_stability')
print('Saved nb30_F4_beta_stability.pdf')

β(None) vs β(full controls) across all KO-metal pairs:
  Spearman ρ = +0.540  p = 1.293e-15  n = 188

β stability (None vs full controls) per metal:
  AS : ρ=+0.788  p=1.054e-06  n=27
  CD : ρ=+0.660  p=2.427e-04  n=26
  CR : ρ=+0.796  p=6.852e-07  n=27
  CU : ρ=+0.002  p=9.904e-01  n=27
  NI : ρ=+0.526  p=4.807e-03  n=27
  PB : ρ=+0.818  p=1.871e-07  n=27
  ZN : ρ=+0.126  p=5.319e-01  n=27


Saved nb30_F4_beta_stability.pdf


In [11]:
# ── Summary: how many KO-metal pairs survive p<0.05 per control combo? ─────────
sig_counts = (
    sweep_lam[sweep_lam['p_value'] < 0.05]
    .groupby('controls')['ko_id'].count()
    .reindex(COMBO_LABELS)
    .fillna(0)
    .astype(int)
)
total_tests = (
    sweep_lam
    .groupby('controls')['ko_id'].count()
    .reindex(COMBO_LABELS)
    .fillna(0)
    .astype(int)
)

print('Significant KO-metal pairs (p<0.05) per control combo:')
print(f'{"Controls":<28s}  {"n_sig":>6s}  {"n_total":>8s}  {"pct":>6s}')
print('-' * 55)
for ctrl in COMBO_LABELS:
    n_s = sig_counts.get(ctrl, 0)
    n_t = total_tests.get(ctrl, 0)
    pct = 100 * n_s / n_t if n_t > 0 else 0
    print(f'{ctrl:<28s}  {n_s:>6d}  {n_t:>8d}  {pct:>5.1f}%')

Significant KO-metal pairs (p<0.05) per control combo:
Controls                       n_sig   n_total     pct
-------------------------------------------------------
None                              12       188    6.4%
pH                                13       188    6.9%
SOM                               12       188    6.4%
Temp                               7       188    3.7%
pH + SOM                          13       188    6.9%
pH + Temp                          8       188    4.3%
SOM + Temp                         7       188    3.7%
pH + SOM + Temp                    9       188    4.8%
|Lat|                              6       188    3.2%
pH + |Lat|                         7       188    3.7%
pH + SOM + |Lat|                   9       188    4.8%
pH + SOM + Temp + |Lat|            9       188    4.8%


In [12]:
# ── Figure 5: n_sig and ρ(λ,|β|) vs control complexity ───────────────────────
fig, axs = plt.subplots(1, 2, figsize=(FIGW['2col'], ROW_H))

# Panel A: n_sig counts
ax = axs[0]
x_pos = range(len(COMBO_LABELS))
n_vals = [sig_counts.get(c, 0) for c in COMBO_LABELS]
ax.bar(x_pos, n_vals, edgecolor='k', linewidth=0.5, color=PALETTE[0])
ax.axhline(len(sweep_lam[sweep_lam['controls'] == 'None']) * 0.05,
           color='gray', lw=0.8, ls='--', label='5% chance level')
ax.set_xticks(x_pos)
ax.set_xticklabels(COMBO_LABELS, rotation=50, ha='right', fontsize=6.5)
ax.set_ylabel('N significant KO-metal pairs (p<0.05)', fontsize=9)
ax.set_title('Signal survival across control combos', fontsize=10)
ax.legend(fontsize=8)
grid_h(ax)

# Panel B: mean |ρ| across metals per control combo
ax = axs[1]
mean_abs_rho = (
    summary
    .groupby('controls')['rho']
    .apply(lambda x: x.abs().mean())
    .reindex(COMBO_LABELS)
)
ax.bar(x_pos, mean_abs_rho.values, edgecolor='k', linewidth=0.5, color=PALETTE[1])
ax.set_xticks(x_pos)
ax.set_xticklabels(COMBO_LABELS, rotation=50, ha='right', fontsize=6.5)
ax.set_ylabel('Mean |ρ(λ, |β|)| across metals', fontsize=9)
ax.set_title('Mean λ-β association strength across controls', fontsize=10)
grid_h(ax)

fig.suptitle('NB30 Fig 5: How controls affect signal and λ-β association', y=1.02)
save(fig, FIGS / 'nb30_F5_control_effect')
print('Saved nb30_F5_control_effect.pdf')

Saved nb30_F5_control_effect.pdf


In [13]:
# ── Text summary of key findings ───────────────────────────────────────────────
print('=' * 60)
print('NB30 KEY FINDINGS')
print('=' * 60)

# 1. Which combos show significant λ-β relationship?
sig_rho = summary[summary['p_value'] < 0.05].sort_values('rho')
print(f'\n1. ρ(λ, |β_USGS|) significant at p<0.05:')
if len(sig_rho) == 0:
    print('   None across any metal × control combo')
else:
    for _, r in sig_rho.iterrows():
        print(f'   {r["metal"].upper()} | {r["controls"]}: ρ={r["rho"]:+.3f} p={r["p_value"]:.3e}')

# 2. Does adding controls consistently change the direction?
print(f'\n2. Mean ρ change: None → full controls')
for metal in USGS_METALS:
    rho_none = summary[(summary['metal'] == metal) & (summary['controls'] == 'None')]['rho'].values
    rho_full = summary[(summary['metal'] == metal) & (summary['controls'] == 'pH + SOM + Temp + |Lat|')]['rho'].values
    if len(rho_none) > 0 and len(rho_full) > 0:
        delta = rho_full[0] - rho_none[0]
        print(f'   {metal.upper():3s}: {rho_none[0]:+.3f} → {rho_full[0]:+.3f}  (Δ={delta:+.3f})')

# 3. β stability
print(f'\n3. β stability (no controls vs full panel):')
for metal in sorted(cmp['metal'].unique()):
    sub = cmp[cmp['metal'] == metal]
    r, p = stats.spearmanr(sub['abs_base'], sub['abs_full'])
    print(f'   {metal.upper():3s}: ρ={r:+.3f} p={p:.3e} n={len(sub)}')

print('\n[End of NB30]')

NB30 KEY FINDINGS

1. ρ(λ, |β_USGS|) significant at p<0.05:
   CU | None: ρ=+0.382 p=4.915e-02
   CU | SOM + Temp: ρ=+0.382 p=4.915e-02
   NI | pH + SOM + Temp: ρ=+0.389 p=4.498e-02
   NI | SOM + Temp: ρ=+0.389 p=4.498e-02
   NI | pH + SOM + |Lat|: ρ=+0.400 p=3.877e-02
   ZN | pH + Temp: ρ=+0.427 p=2.619e-02
   NI | pH + SOM + Temp + |Lat|: ρ=+0.428 p=2.595e-02
   CU | |Lat|: ρ=+0.432 p=2.457e-02
   ZN | pH + SOM + Temp: ρ=+0.436 p=2.304e-02
   PB | pH + SOM: ρ=+0.440 p=2.178e-02
   PB | SOM: ρ=+0.476 p=1.217e-02
   CU | pH: ρ=+0.481 p=1.107e-02
   ZN | pH + SOM + Temp + |Lat|: ρ=+0.482 p=1.084e-02
   ZN | SOM + Temp: ρ=+0.531 p=4.363e-03

2. Mean ρ change: None → full controls
   AS : +0.092 → +0.005  (Δ=-0.087)
   CD : +0.196 → -0.102  (Δ=-0.297)
   CR : -0.158 → -0.187  (Δ=-0.029)
   CU : +0.382 → +0.340  (Δ=-0.042)
   NI : +0.283 → +0.428  (Δ=+0.145)
   PB : +0.336 → +0.180  (Δ=-0.156)
   ZN : +0.221 → +0.482  (Δ=+0.261)

3. β stability (no controls vs full panel):
   AS : ρ=+0.788